<a href="https://colab.research.google.com/github/mundundan-star/online-retail-customer-analytics/blob/main/2.0%20Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The primary purpose of this notebook is to control what features are fed in a particular notebook\model.
To ensure convenient feature_engineering from transaction items, particularly for machine learning training and production, python functions were created and stored in an AWS S3 bucket. This is to esnure that all notebooks beyond this one, only pull data from the database through a consistent channel, using the `v_clean_sales_analytics` via the functions tailored each notebook's needs. That said, this notebooks purpose is the creation of a `feature_engineering` function that will deliver features for revenue, and customer churn, prediction, as well a `customer_segmentation_features` which is depended on data produced by the features engineering, to serve the customer segmentation training and assignments.

In [ ]:
!pip install boto3
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import userdata

from sqlalchemy import create_engine, text, inspect

import boto3
import sys
import os
import importlib

bucket_name = 'sales-data-analytics-portfolio-2026'

#Intializing s3 client with credentials
s3 = boto3.client(
    "s3",
    aws_access_key_id = userdata.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key = userdata.get('AWS_SECRET_ACCESS_KEY'),
    region_name = 'eu-north-1'
)

with open("feature_engineering.py", "w") as f:
    f.write("""
import pandas as pd
import numpy as np
def feature_engineering(df):

    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
    end_date = df["InvoiceDate"].max()

    cust_data = df.groupby("CustomerID")[["InvoiceDate", "StockCode", "Quantity", "UnitPrice", "Revenue"]].agg(
       FirstPurchase = ("InvoiceDate", "min"),
       LastPurchase = ("InvoiceDate", "max"),
       Frequency = ("InvoiceDate", "nunique"),
       ProductDiversity = ("StockCode", "nunique"),
       AvgUnitPrice = ("UnitPrice", "mean"),
       AvgQuantity = ("Quantity", "mean"),
       AOV = ("Revenue", "mean"),
       TotalRevenue = ("Revenue", "sum")
    )

    cust_data["CohortMonth"] = cust_data["FirstPurchase"].dt.to_period("M")
    cust_data["Recency"] = 1 + (end_date - cust_data["LastPurchase"]).dt.days
    cust_data["Tenure"] = 1 + (end_date - cust_data["FirstPurchase"]).dt.days
    cust_data["ObservedLifeSpan"] = 1 + (cust_data["LastPurchase"] - cust_data["FirstPurchase"]).dt.days

    cust_data.drop(["FirstPurchase","LastPurchase"], axis = 1, inplace = True)

    cust_data["RecencyToTenure"] = cust_data["Recency"]/(cust_data["Tenure"])
    cust_data["ActivePurchaseDensity"] = cust_data["Frequency"]/cust_data["ObservedLifeSpan"]
    cust_data["LifetimePurchaseDensity"] = cust_data["Frequency"]/cust_data["Tenure"]

    cust_data["ActiveMRate"] = cust_data["TotalRevenue"]/cust_data["ObservedLifeSpan"]
    cust_data["LifetimeMRate"] = cust_data["TotalRevenue"]/cust_data["Tenure"]

    ipi = (df[["CustomerID", "InvoiceDate"]].sort_values(["CustomerID", "InvoiceDate"], ascending = False))
    ipi = ipi.groupby(["CustomerID", "InvoiceDate"]).size().reset_index()
    ipi["InterPurchaseInterval"] = np.where(ipi["CustomerID"].shift(1) == ipi["CustomerID"], (ipi["InvoiceDate"] - ipi["InvoiceDate"].shift(1)).dt.days + 1, 1)
    ipi = ipi.groupby("CustomerID").agg(
        AvgIPI = ("InterPurchaseInterval", "mean")
    ).reset_index()

    cust_data = cust_data.merge(ipi[["CustomerID", "AvgIPI"]], on = "CustomerID", how = "left")

    cust_data["RelativeSilence"] = cust_data["Recency"]/cust_data["AvgIPI"]

    return cust_data"""
)


# Uploading file to s3 bucket
s3.upload_file('feature_engineering.py', bucket_name, 'functions/feature_engineering.py')
print(f'Successfully uploaded file to {bucket_name} bucket')

s3.download_file(bucket_name, 'functions/feature_engineering.py', 'feature_engineering.py')

sys.path.append(os.getcwd())

import feature_engineering

importlib.reload(feature_engineering)

from feature_engineering import feature_engineering

engine = create_engine(userdata.get("NEON_DATABASE_URL"))

query = """
SELECT *
FROM v_clean_sales_analytics
"""

df = pd.read_sql(text(query), con = engine)

df0 = feature_engineering(df)
print('Preview of Features')
print(df0.info())
df0

Successfully uploaded file to sales-data-analytics-portfolio-2026 bucket
Preview of Features
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4339 entries, 0 to 4338
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype    
---  ------                   --------------  -----    
 0   CustomerID               4339 non-null   float64  
 1   Frequency                4339 non-null   int64    
 2   ProductDiversity         4339 non-null   int64    
 3   AvgUnitPrice             4339 non-null   float64  
 4   AvgQuantity              4339 non-null   float64  
 5   AOV                      4339 non-null   float64  
 6   TotalRevenue             4339 non-null   float64  
 7   CohortMonth              4339 non-null   period[M]
 8   Recency                  4339 non-null   int64    
 9   Tenure                   4339 non-null   int64    
 10  ObservedLifeSpan         4339 non-null   int64    
 11  RecencyToTenure          4339 non-null   float64  
 12  ActivePurch

,CustomerID,Frequency,ProductDiversity,AvgUnitPrice,AvgQuantity,AOV,TotalRevenue,CohortMonth,Recency,Tenure,ObservedLifeSpan,RecencyToTenure,ActivePurchaseDensity,LifetimePurchaseDensity,ActiveMRate,LifetimeMRate,AvgIPI,RelativeSilence
0,12346.0,1,1,1.040000,74215.000000,77183.600000,77183.60,2011-01,326,326,1,1.000000,1.000000,0.003067,77183.600000,236.759509,1.000000,326.000000
1,12347.0,7,103,2.644011,13.505495,23.681319,4310.00,2010-12,3,368,366,0.008152,0.019126,0.019022,11.775956,11.711957,53.142857,0.056452
2,12348.0,4,22,5.764839,75.516129,57.975484,1797.24,2010-12,76,359,284,0.211699,0.014085,0.011142,6.328310,5.006240,71.750000,1.059233
3,12349.0,1,73,8.289041,8.643836,24.076027,1757.55,2011-11,19,19,1,1.000000,1.000000,0.052632,1757.550000,92.502632,1.000000,19.000000
4,12350.0,1,17,3.841176,11.588235,19.670588,334.40,2011-02,311,311,1,1.000000,1.000000,0.003215,334.400000,1.075241,1.000000,311.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4334,18280.0,1,10,4.765000,4.500000,18.060000,180.60,2011-03,278,278,1,1.000000,1.000000,0.003597,180.600000,0.649640,1.000000,278.000000
4335,18281.0,1,7,5.622857,7.714286,11.545714,80.82,2011-06,181,181,1,1.000000,1.000000,0.005525,80.820000,0.446519,1.000000,181.000000
4336,18282.0,2,12,5.199167,8.583333,14.837500,178.05,2011-08,8,127,120,0.062992,0.016667,0.015748,1.483750,1.401969,60.500000,0.132231
4337,18283.0,14,263,1.628752,1.882108,2.837074,2045.53,2011-01,4,338,335,0.011834,0.041791,0.041420,6.106060,6.051864,24.857143,0.160920


In [ ]:
# creating function to pull features specifically for customer Segmentation from dataframe after passing feature_engineering function

import sys
import os

with open("segmentation_features.py", "w") as f:
    f.write("""def segmentation_features(df):
    features = [
    "CustomerID",
    "Tenure",
    "ObservedLifeSpan",
    "TotalRevenue",
    "Recency",
    "ProductDiversity",
    "Frequency"]

    return df[features]""")

s3.upload_file("segmentation_features.py", bucket_name, "functions/segmentation_features.py")
print("Successfully uploaded Segmentation function to S3 Bucket")

s3.download_file(bucket_name, "functions/segmentation_features.py", "segmentation_features.py")
print("\nSuccessfully download Segmentation function from AWA S3")

sys.path.append(os.getcwd())
import segmentation_features

importlib.reload(segmentation_features)

from segmentation_features import segmentation_features

df1 = segmentation_features(df0)
print("\nPreview of features particular to Customer Segmentation")
df1.head()

Successfully uploaded Segmentation function to S3 Bucket

Successfully download Segmentation function from AWA S3

Preview of features particular to Customer Segmentation


,CustomerID,Tenure,ObservedLifeSpan,TotalRevenue,Recency,ProductDiversity,Frequency
0,12346.0,326,1,77183.60,326,1,1
1,12347.0,368,366,4310.00,3,103,7
2,12348.0,359,284,1797.24,76,22,4
3,12349.0,19,1,1757.55,19,73,1
4,12350.0,311,1,334.40,311,17,1


In [ ]:
#Feature Engineering specific to Churn Predictions Notebook

with open("churn_label_assignment.py", "w") as f:
    f.write("""import pandas as pd
import numpy as np
def churn_label_assignment(df):
        data1 = df[df["InvoiceDate"] <= '2011-08-31']
        data2 = df[df["InvoiceDate"] > '2011-08-31']

        # Last purchase before 2011-08-31
        lastpurchase = data1.groupby("CustomerID").agg(
        LastPurchase = ("InvoiceDate","max"))

        # First purchase after 2011-08-31
        firstpurchase = data2.groupby("CustomerID").agg(
        FirstPurchase = ("InvoiceDate","min"))

        data = lastpurchase.join(firstpurchase, how = 'left')

        cutoff_date = data1["InvoiceDate"].max()

        # InterPurchaseInterval to know if customer had churned by August custoff
        data["InterPurchaseInterval1"] = (cutoff_date - data["LastPurchase"]).dt.days

        # InterPurchaseInterval to determine churn after August cutoff
        data["InterPurchaseInterval2"] = (data["FirstPurchase"] - data["LastPurchase"]).dt.days

        churn_days = 100
        data["Inactive"] = np.where(data["InterPurchaseInterval1"] < churn_days, 0, 1)
        data["Churned"] = np.where(data["InterPurchaseInterval2"] < churn_days, 0, 1)

        data = data["Churned"].reset_index()

        print()
        print("Churn labels for transactions before 2011-08-31")
        return data""")

s3.upload_file("churn_label_assignment.py", bucket_name, "functions/churn_label_assignment.py")
print("Successfully uploaded churn function to AWS S3 bucket")

s3.download_file(bucket_name, "functions/churn_label_assignment.py", "churn_label_assignment")
print("Successfully downloaded churn function from AWS S3 bucket for preview")

sys.path.append(os.getcwd())
import churn_label_assignment
importlib.reload(churn_label_assignment)
from churn_label_assignment import churn_label_assignment

churn_label_assignment(df).head()

Successfully uploaded churn function to AWS S3 bucket
Successfully downloaded churn function from AWS S3 bucket for preview

Churn labels for transactions before 2011-08-31


,CustomerID,Churned
0,12346.0,1
1,12347.0,0
2,12348.0,1
3,12350.0,1
4,12352.0,1
